# Silver Layer — Cleaned & Validated Data

Silver layer transformations:
- **PII Masking**: Email, phone, name hashed with SHA-256
- **Deduplication**: Window-based row_number dedup
- **Data Quality**: `dq_is_valid` flag, `dq_reason` column

In [1]:
%pip install pyspark==3.5.3

  Using cached pyspark-3.5.3-py2.py3-none-any.whl
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.0
    Can't uninstall 'pyspark'. No files were found to uninstall.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .appName("explore_silver") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://dataops-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataops-key") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataops-secret") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print(f"Spark {spark.version} ready")

Spark 3.5.0 ready


In [3]:
BASE = "s3a://delta-lake/silver"
tables = ["customers", "products", "transactions"]
for t in tables:
    df = spark.read.format("delta").load(f"{BASE}/{t}")
    print(f"\n{'='*60}")
    print(f"{t.upper()}: {df.count()} rows \u00d7 {len(df.columns)} cols")
    df.printSchema()
    df.show(5, truncate=False)


CUSTOMERS: 100 rows × 18 cols
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email_masked: string (nullable = true)
 |-- phone_masked: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- dq_is_valid: boolean (nullable = true)
 |-- dq_validation_errors: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _load_date: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _created_at: timestamp (nullable = true)
 |-- _updated_at: timestamp (nullable = true)

+-----------+----------+---------+----------------------------------------------------------------+----------------------------------------------------------------+------------+

In [4]:
# PII masking verification
df = spark.read.format("delta").load(f"{BASE}/customers")
print("Email looks hashed:")
df.select("email_masked").show(5, truncate=False)
print("Phone looks hashed:")
df.select("phone_masked").show(5, truncate=False)

Email looks hashed:
+----------------------------------------------------------------+
|email_masked                                                    |
+----------------------------------------------------------------+
|6d78780a411297600e6a764c48085c0a4f23a9d8bb34f1dd4a92058f0d2c6038|
|7e915c54ffdcde3f327222af4f9ab2b3e132726bbf7681e8bd8435c33c4bbf79|
|9aceb9a72514854abab068dc1491eed59e49f6a03de7e1690fe4282f2dc21cc8|
|246d53ab793b42726a77d16cdeaae42bbc0b282b6a354617da9b2784e1a725c7|
|852189bfe65e5adb528d65048e5f021ac80b123050e3f653d5bc551b75df6871|
+----------------------------------------------------------------+
only showing top 5 rows

Phone looks hashed:
+----------------------------------------------------------------+
|phone_masked                                                    |
+----------------------------------------------------------------+
|8ab3424f4befd897ba6e6cdf6e51537726df0b0c2215668bb7ebaf1103ca266d|
|dfd9c4588e7924c1a9634ec80db0a6626987b72b8fc72668bd81d3cb6136f64

In [5]:
# Data quality distribution
for t in tables:
    df = spark.read.format("delta").load(f"{BASE}/{t}")
    print(f"\n{t.upper()} DQ stats:")
    df.groupBy("dq_is_valid").count().show()


CUSTOMERS DQ stats:
+-----------+-----+
|dq_is_valid|count|
+-----------+-----+
|       true|  100|
+-----------+-----+


PRODUCTS DQ stats:
+-----------+-----+
|dq_is_valid|count|
+-----------+-----+
|       true|   50|
+-----------+-----+


TRANSACTIONS DQ stats:
+-----------+-----+
|dq_is_valid|count|
+-----------+-----+
|       true|  100|
+-----------+-----+



In [6]:
# Compare bronze vs silver row counts
BRONZE = "s3a://delta-lake/bronze"
for t in tables:
    b = spark.read.format("delta").load(f"{BRONZE}/{t}").count()
    s = spark.read.format("delta").load(f"{BASE}/{t}").count()
    print(f"{t}: bronze={b}  silver={s}  diff={b-s}")

customers: bronze=100  silver=100  diff=0
products: bronze=50  silver=50  diff=0
transactions: bronze=100  silver=100  diff=0


In [7]:
spark.stop()